# Lecture 10: Training Neural Networks I

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 10, with material from Stanford CS231n 2026, Lecture 6.*

Lectures 10 and 11 are the practical "how to actually train a network" lectures. This one covers the **one-time setup** you do before training starts.

**What you'll learn**
1. Activation functions (a quick recap).
2. **Data preprocessing**: centering and normalizing.
3. **Weight initialization**: why it matters, and Xavier/Kaiming initialization.
4. **Regularization**: dropout, and the "add randomness, then average it out" pattern.
5. **Data augmentation**, cutout, and mixup.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.datasets import load_digits

## 1. Activation functions (recap)

From Lecture 5: **ReLU** is the default; sigmoid and tanh **saturate** (their slope is ~0 for large inputs, which kills gradients); GELU is the default in Transformers.

One more problem with sigmoid: its outputs are **always positive**. Suppose a neuron $f = \sum_i w_ix_i + b$ gets inputs $x_i$ that are all positive. Its weight gradients are $\frac{\partial L}{\partial w_i} = \frac{\partial L}{\partial f}\cdot x_i$, so they **all have the same sign**: on every step, all the weights go up together or down together. Reaching weights with mixed signs takes an inefficient zig-zag path.

In [ ]:
x_positive = np.array([0.2, 0.7, 0.5, 0.9])     # e.g. sigmoid outputs: all positive
for upstream in [+1.5, -0.8]:
    print(f"upstream gradient {upstream:+}: weight gradients {upstream * x_positive}")

This is one reason we want **zero-centered** data, which brings us to preprocessing.

## 2. Data preprocessing

Two standard steps, using statistics from the **training set only**:
* **zero-center:** subtract the mean of each feature;
* **normalize:** divide by the standard deviation of each feature.

Here's a 2-D example:

In [ ]:
rng = np.random.default_rng(1)
data = rng.multivariate_normal(mean=[5, 3], cov=[[3, 2.2], [2.2, 2]], size=500)

centered = data - data.mean(axis=0)
normalized = centered / centered.std(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
for ax, points, title in [(axes[0], data, "original"), (axes[1], centered, "zero-centered"), (axes[2], normalized, "normalized")]:
    ax.scatter(points[:, 0], points[:, 1], s=3)
    ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
    ax.set_title(title); ax.set_aspect("equal")
plt.show()
print("mean after centering:", centered.mean(axis=0).round(3) + 0.0)
print("std after normalizing:", normalized.std(axis=0).round(3))

**Why does it help?** Before normalization, the classification loss is very sensitive to tiny changes in the weights (a small tilt of the decision boundary far from the origin changes a lot of predictions), which makes optimization hard. After normalization, the loss is much better behaved.

(There are fancier versions, **PCA** to decorrelate the features and **whitening** to make the covariance the identity, but they're rarely used for images.)

**For images, the common choices are:**
* subtract the **mean image** (AlexNet): an array the size of an image;
* subtract the **mean of each color channel** (VGG): just 3 numbers;
* subtract the per-channel mean and **divide by the per-channel std** (ResNet): the most common today. For ImageNet, those are mean (0.485, 0.456, 0.406) and std (0.229, 0.224, 0.225).

## 3. Weight initialization

**Q: What if every weight starts at the same value, e.g., 0?** Then every hidden unit computes the same thing and gets the same gradient, so they stay identical forever. We need **random** initial weights. But how big?

### Johnson's experiment
A **6-layer network** with **tanh** activations and 4,096 units per layer. We feed in random data and look at the activations in each layer. First, weights drawn as `0.01 * randn`:

In [ ]:
def activation_stats(weight_scale, activation=np.tanh, n_layers=6, width=4096):
    # Push random data through n_layers random layers; return each layer's activations.
    rng = np.random.default_rng(0)
    h = rng.normal(size=(256, width))
    layers = []
    for layer in range(n_layers):
        W = weight_scale * rng.normal(size=(width, width))
        h = activation(h @ W)
        layers.append(h)
    return layers

def plot_layers(layers, title):
    fig, axes = plt.subplots(1, len(layers), figsize=(14, 2))
    for i, h in enumerate(layers):
        axes[i].hist(h.ravel(), bins=50, range=(-1, 1))
        axes[i].set_title(f"layer {i + 1}\nstd {h.std():.3f}", fontsize=8)
        axes[i].set_yticks([])
    plt.suptitle(title, y=1.15)
    plt.show()

plot_layers(activation_stats(0.01), "weights = 0.01 * randn")

**The activations shrink toward zero, layer after layer.** *Q: What do the gradients look like?* Each layer's weight gradient is (its input) × (upstream gradient), and both are tiny: **little or no learning.**

Now try weights a bit bigger, `0.05 * randn`:

In [ ]:
plot_layers(activation_stats(0.05), "weights = 0.05 * randn")

**Now the activations pile up at −1 and +1: tanh is saturated.** Its slope there is ~0, so the local gradients are ~0: again, **learning breaks down**.

### Xavier initialization
**Xavier** (Glorot & Bengio, 2010): use `randn / sqrt(D_in)`, where $D_{in}$ is the number of inputs to the layer.

In [ ]:
plot_layers(activation_stats(1 / np.sqrt(4096)), "Xavier: weights = randn / sqrt(D_in)")

**Nicely spread out in every layer.** Why does $1/\sqrt{D_{in}}$ work? One output of a layer is $y = \sum_{j=1}^{D_{in}}x_jw_j$. If the $x$'s and $w$'s are independent with mean 0:

$$\text{Var}(y) = D_{in}\cdot\text{Var}(x)\cdot\text{Var}(w)$$

So choosing $\text{Var}(w) = 1/D_{in}$ makes $\text{Var}(y) = \text{Var}(x)$: **the variance going out equals the variance coming in.** A quick numerical check:

In [ ]:
D_in = 512
x = rng.normal(0, 2.0, size=(2000, D_in))                  # Var(x) = 4
W = rng.normal(0, np.sqrt(1 / D_in), size=(D_in, 256))       # Var(w) = 1 / D_in
print(f"Var(x) = {x.var():.2f},  Var(y) = {(x @ W).var():.2f}  <- the same")

### What about ReLU? Kaiming initialization
Xavier assumes a zero-centered activation like tanh. **ReLU sets half of its inputs to zero, which halves the variance**, so with Xavier the activations shrink again with every layer. **Kaiming / MSRA initialization** (He et al., 2015) doubles the variance to make up for it: `randn * sqrt(2 / D_in)`.

In [ ]:
relu = lambda z: np.maximum(0, z)
xavier_relu = activation_stats(1 / np.sqrt(4096), activation=relu)
kaiming_relu = activation_stats(np.sqrt(2 / 4096), activation=relu)
print("ReLU + Xavier,  std per layer:", [round(float(h.std()), 3) for h in xavier_relu], " <- shrinking")
print("ReLU + Kaiming, std per layer:", [round(float(h.std()), 3) for h in kaiming_relu], " <- steady")

**Rules of thumb:** Xavier for tanh, **Kaiming for ReLU** (the usual case).

**Residual networks:** with Kaiming init, each residual block adds its own variance on top of its input's, so the variance grows from block to block. The fix: **initialize the last layer of each residual block to zero**, so every block starts as the identity.

## 4. Regularization

**Your model trains, but it overfits.** Lecture 3 covered regularization terms like L2. Another family of regularizers **adds randomness during training**.

### Dropout
**In each forward pass, randomly set some neurons to zero** (a common rate is half of them). Why would that help?
* It forces the network to have **redundant representations**: no unit can rely on one particular other unit always being there.
* Another view: dropout trains a huge **ensemble** of networks that share weights (each random mask is a different network).

**At test time** we want a fixed, non-random output. If each unit was kept with probability $p$ during training, its average output was $p$ times its full value. The common implementation, **inverted dropout**, divides by $p$ during training, so test time needs no change at all:

In [ ]:
def dropout(h, keep_prob, rng):
    # Training-time inverted dropout: zero out units at random, scale the survivors up by 1/keep_prob.
    mask = (rng.random(h.shape) < keep_prob) / keep_prob
    return h * mask

h = np.array([1.0, 2.0, 3.0, 4.0])
for i in range(3):
    print(f"random training pass {i + 1}:", dropout(h, 0.5, rng))

average = np.mean([dropout(h, 0.5, rng) for _ in range(10000)], axis=0)
print("average over 10,000 passes:", average.round(2), " <- matches the test-time output", h)

**Does it reduce overfitting?** Train a fairly big network (64 → 256 → 256 → 10) on only **300** digits, with and without dropout. First the data and a simple training function:

In [ ]:
digits = load_digits()
images = digits.images / 16.0 - 0.3
labels = digits.target
order = np.random.default_rng(0).permutation(len(images))
train_idx, val_idx = order[:300], order[1000:1300]
X_train, y_train = images[train_idx].reshape(-1, 64), labels[train_idx]
X_val, y_val = images[val_idx].reshape(-1, 64), labels[val_idx]

In [ ]:
def train_network(X, y, keep_prob=1.0, augment=None, epochs=150, lr=0.02, seed=0):
    # A 64-256-256-10 ReLU network trained with SGD + momentum. Returns a predict function.
    rng = np.random.default_rng(seed)
    sizes = [64, 256, 256, 10]
    Ws = [rng.normal(0, np.sqrt(2 / a), size=(a, b)) for a, b in zip(sizes[:-1], sizes[1:])]   # Kaiming init
    bs = [np.zeros(b) for b in sizes[1:]]
    vW = [np.zeros_like(W) for W in Ws]; vb = [np.zeros_like(b) for b in bs]
    for epoch in range(epochs):
        for batch in np.array_split(rng.permutation(len(X)), len(X) // 50):
            Xb = X[batch] if augment is None else augment(X[batch], rng)
            # forward, remembering each layer's input, pre-activation, and dropout mask
            h, saved = Xb, []
            for i in range(3):
                z = h @ Ws[i] + bs[i]
                if i < 2:
                    mask = (rng.random(z.shape) < keep_prob) / keep_prob
                    saved.append((h, z, mask))
                    h = np.maximum(0, z) * mask
                else:
                    saved.append((h, z, None))
                    scores = z
            probs = np.exp(scores - scores.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
            probs[np.arange(len(batch)), y[batch]] -= 1
            d = probs / len(batch)
            # backward and update
            for i in reversed(range(3)):
                h_in, z, mask = saved[i]
                if i < 2:
                    d = d * mask * (z > 0)                    # back through dropout and ReLU
                gW, gb = h_in.T @ d, d.sum(0)
                d = d @ Ws[i].T
                vW[i] = 0.9 * vW[i] - lr * gW; Ws[i] += vW[i]
                vb[i] = 0.9 * vb[i] - lr * gb; bs[i] += vb[i]
    def predict(X_new):
        h = X_new
        for i in range(2):
            h = np.maximum(0, h @ Ws[i] + bs[i])             # test time: no dropout
        return np.argmax(h @ Ws[2] + bs[2], axis=1)
    return predict

In [ ]:
for name, keep_prob in [("no dropout", 1.0), ("dropout (keep half)", 0.5)]:
    train_accs, val_accs = [], []
    for seed in range(3):
        predict = train_network(X_train, y_train, keep_prob=keep_prob, seed=seed)
        train_accs.append(np.mean(predict(X_train) == y_train))
        val_accs.append(np.mean(predict(X_val) == y_val))
    print(f"{name:20s}: train {np.mean(train_accs):.1%}, validation {np.mean(val_accs):.1%}  (average of 3 runs)")

**What we see:** here dropout makes almost no difference: both versions fit the training digits perfectly, and validation accuracy is about the same. These digits are easy enough that the network generalizes well either way. Dropout helps most with **large fully connected layers on harder data**: AlexNet and VGG used it in their huge FC layers. Later architectures like ResNet, which have no big FC layers, mostly don't use it. (Try it: train on only 100 digits instead of 300.)

### A common pattern
**Training: add some randomness. Testing: average it out.** Dropout follows this pattern, and so does **batch normalization** (random minibatch statistics in training, fixed statistics at test time; Lecture 7). For ResNet-style networks, BN is often the only regularizer needed.

## 5. Data augmentation

**Transform the training images randomly in ways that don't change their label.** The network sees a new variation of each image every epoch. Common augmentations for photos:
* **horizontal flips** (a mirrored cat is still a cat);
* **random crops and scales** (ResNet: resize so the short side is a random length between 256 and 480, then take a random 224×224 crop);
* **color jitter** (random brightness and contrast).

**The augmentation must preserve the label**, and that depends on your data: horizontal flips are standard for photos but **wrong for digits** (a mirrored 3 isn't a 3). For digits, small **shifts** are safe:

In [ ]:
def random_shifts(batch, rng):
    # Shift each 8x8 image by up to 1 pixel in each direction.
    out = np.empty_like(batch)
    for k, flat_image in enumerate(batch):
        shift = rng.integers(-1, 2, size=2)
        shifted = ndimage.shift(flat_image.reshape(8, 8), shift, order=0, cval=-0.3)
        out[k] = shifted.ravel()
    return out

example = X_train[:1]
fig, axes = plt.subplots(1, 6, figsize=(8, 1.6))
axes[0].imshow(example.reshape(8, 8), cmap="gray_r"); axes[0].set_title("original", fontsize=8)
for k in range(1, 5):
    axes[k].imshow(random_shifts(example, rng).reshape(8, 8), cmap="gray_r"); axes[k].set_title("shifted", fontsize=8)
axes[5].imshow(example.reshape(8, 8)[:, ::-1], cmap="gray_r"); axes[5].set_title("flipped: wrong!", fontsize=8)
for ax in axes: ax.axis("off")
plt.show()

Train with and without shift augmentation, and test on normal validation digits **and** validation digits shifted by one pixel:

In [ ]:
X_val_shifted = np.array([ndimage.shift(img.reshape(8, 8), (0, 1), order=0, cval=-0.3).ravel() for img in X_val])

for name, augment in [("no augmentation", None), ("random shifts", random_shifts)]:
    predict = train_network(X_train, y_train, augment=augment)
    print(f"{name:16s}: normal validation {np.mean(predict(X_val) == y_val):.1%},  "
          f"shifted validation {np.mean(predict(X_val_shifted) == y_val):.1%}")

**What we see:** augmentation makes the network **robust to shifts**: accuracy on shifted digits jumps a lot. On the *normal* validation set it costs a little here, because these digits were all carefully centered when the dataset was made, so the clean test set contains none of the variation we taught. In real photos, objects do appear in different positions, which is why augmentation like random crops usually **improves** test accuracy. **Augment with the variations your test data actually has.**

## 6. More regularizers that add randomness

* **DropConnect:** drop random *weights* instead of activations.
* **Stochastic depth:** randomly skip whole residual blocks during training.
* **Cutout:** set a random square of each training image to zero.
* **Mixup:** train on **blends** of two images, with their labels blended the same way (e.g., 40% cat + 60% dog → target "0.4 cat, 0.6 dog").

In [ ]:
def cutout(image, rng, size=3):
    out = image.copy()
    top, left = rng.integers(0, 8 - size + 1, size=2)
    out[top:top + size, left:left + size] = 0
    return out

def mixup(image_a, image_b, label_a, label_b, rng):
    lam = rng.beta(1.0, 1.0)                      # random mixing weight between 0 and 1
    target = np.zeros(10)
    target[label_a] += lam
    target[label_b] += 1 - lam
    return lam * image_a + (1 - lam) * image_b, target

img_a, img_b = images[train_idx[0]], images[train_idx[1]]
mixed, target = mixup(img_a, img_b, labels[train_idx[0]], labels[train_idx[1]], rng)
fig, axes = plt.subplots(1, 3, figsize=(5, 1.8))
axes[0].imshow(cutout(img_a, rng), cmap="gray_r"); axes[0].set_title("cutout", fontsize=8)
axes[1].imshow(mixed, cmap="gray_r"); axes[1].set_title("mixup", fontsize=8)
axes[2].bar(range(10), target); axes[2].set_title("mixup target", fontsize=8)
for ax in axes[:2]: ax.axis("off")
plt.show()

**Johnson's advice:** consider dropout for large fully connected layers; **batch normalization and data augmentation are almost always a good idea**; try cutout and mixup, especially for small datasets.

## 7. In PyTorch (optional, run in Colab)

In [ ]:
# [PyTorch] Needs: pip install torch torchvision
import torch.nn as nn
import torchvision.transforms as T

layer = nn.Linear(4096, 4096)
nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")      # Kaiming initialization
dropout_layer = nn.Dropout(p=0.5)                                # NOTE: p is the probability of DROPPING a unit

train_transform = T.Compose([
    T.RandomResizedCrop(224),                                    # random crops and scales
    T.RandomHorizontalFlip(),                                    # flips (fine for photos!)
    T.ColorJitter(0.4, 0.4, 0.4),                                # color jitter
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),   # per-channel mean/std (ResNet-style)
])
print(train_transform)

## Summary
* **Activations:** use ReLU (GELU in Transformers).
* **Preprocessing:** zero-center and normalize, using **training-set** statistics. Images: per-channel mean and std.
* **Initialization:** too small → activations vanish; too large → saturation. **Xavier** ($1/\sqrt{D_{in}}$) for tanh, **Kaiming** ($\sqrt{2/D_{in}}$) for ReLU.
* **Dropout:** randomly zero units during training; inverted dropout scales by $1/p$ so testing needs no change.
* **Pattern:** add randomness in training, average it out at test time.
* **Augmentation:** label-preserving transformations that match the variation in your real data.

### Check your understanding
1. What goes wrong if all weights start at 0?
2. Why does Xavier initialization use $1/\sqrt{D_{in}}$?
3. Why does ReLU need Kaiming instead of Xavier?
4. Why are horizontal flips good for ImageNet but bad for digits?

<details><summary>Answers</summary>

1. Every unit in a layer computes the same thing and gets the same gradient, so they stay identical forever.
2. $\text{Var}(y) = D_{in}\text{Var}(x)\text{Var}(w)$, so $\text{Var}(w) = 1/D_{in}$ keeps the variance the same from layer to layer.
3. ReLU zeroes half its inputs, halving the variance; doubling the weight variance makes up for it.
4. A mirrored photo still shows the same object, but a mirrored digit can look like a different symbol.
</details>

**Next:** Lecture 11, training neural networks (part 2).